# 02 文字检测可视化任务（使用 LabelMe 标注）

本 notebook 不再使用 OpenCV 猜文字区域，也不使用 PaddleOCR/YOLO。

它会读取 `object_detection/data` 中每张图片对应的 LabelMe `.json` 文件，提取标注框，并生成每类 landmark 2 组“检索-检测”可视化结果，共 24 组。

如果某张 query 或检索结果没有对应的 LabelMe json，会在图上标注 `NO JSON`，这说明该图片没有找到人工标注文件，不代表代码坏了。

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from labelme_utils import LANDMARKS, build_labelme_index, generate_24_detection_groups

DATA_DIR = PROJECT_ROOT / "object_detection" / "data"
RETRIEVAL_CSV = PROJECT_ROOT / "outputs" / "retrieval_results" / "retrieval_topk.csv"
DETECTION_OUT = PROJECT_ROOT / "outputs" / "detection_results" / "visual_groups"

print("Data dir:", DATA_DIR)
print("Retrieval CSV:", RETRIEVAL_CSV)
print("Detection output:", DETECTION_OUT)
print("Landmarks:", LANDMARKS)

assert DATA_DIR.exists(), "object_detection/data not found."
assert RETRIEVAL_CSV.exists(), "Please run 01_image_retrieval.ipynb first to generate retrieval_topk.csv."

Data dir: d:\CODE\PY\BJTU_Vision\object_detection\data
Retrieval CSV: d:\CODE\PY\BJTU_Vision\outputs\retrieval_results\retrieval_topk.csv
Detection output: d:\CODE\PY\BJTU_Vision\outputs\detection_results\visual_groups
Landmarks: ['fhy', 'jx', 'kx', 'mh', 'nm', 'sjz', 'sy', 'tsg', 'ty', 'yf', 'yk', 'zx']


## 检查 LabelMe 标注

这里会扫描 `object_detection/data` 下的 `.json` 文件，并建立“图片文件名 -> 标注框”的索引。

In [2]:
annotation_index = build_labelme_index(DATA_DIR)
print("LabelMe json files indexed:", len(annotation_index))
for i, (k, v) in enumerate(annotation_index.items()):
    if i >= 5:
        break
    print(k, "->", v["json_path"].name, "| image:", v["image_path"].name if v.get("image_path") else None)

LabelMe json files indexed: 1494
fhy-0120240508151840 -> fhy-0120240508151840.json | image: fhy-0120240508151840.jpg
fhy-04qgjnsjosp902psvnz -> fhy-04qgjnsjosp902psvnz.json | image: fhy-04qgjnsjosp902psvnz.png
fhy-0589095011 -> fhy-0589095011.json | image: fhy-0589095011.png
fhy-08ur03qu0ridjewifi -> fhy-08ur03qu0ridjewifi.json | image: fhy-08ur03qu0ridjewifi.jpg
fhy-09324032u092r9w -> fhy-09324032u092r9w.json | image: fhy-09324032u092r9w.jpg


## 生成 24 组“检索-检测”结果

每类 landmark 选 2 个 query case。每组图包含：Query + 若干 Top 检索结果。能匹配到 LabelMe json 的图片会画出红色文字标注框。

In [3]:
topk_df = pd.read_csv(RETRIEVAL_CSV)
review_df, review_csv = generate_24_detection_groups(
    topk_df=topk_df,
    data_dir=DATA_DIR,
    out_dir=DETECTION_OUT,
    cases_per_label=2,
    retrieved_per_case=10
)
print("Generated visual rows:", len(review_df))
print("Manual review CSV:", review_csv)
review_df.head(12)

Generated visual rows: 264
Manual review CSV: d:\CODE\PY\BJTU_Vision\outputs\detection_results\manual_review.csv


,case_id,label,visualization_path,role,original_path,shown_path,annotation_found,box_count,json_path,manual_result,manual_note
0,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Query,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,True,1,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,,
1,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Top1,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,False,0,,,
2,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Top2,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,True,1,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,,
3,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Top3,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,False,0,,,
4,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Top4,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,True,1,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,,
5,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Top5,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,True,1,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,,
6,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Top6,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,True,1,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,,
7,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Top7,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,True,1,d:\CODE\PY\BJTU_Vision\object_detection\data\f...,,
8,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Top8,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,False,0,,,
9,fhy_case01,fhy,d:\CODE\PY\BJTU_Vision\outputs\detection_resul...,Top9,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,d:\CODE\PY\BJTU_Vision\object_detection\data\y...,True,1,d:\CODE\PY\BJTU_Vision\object_detection\data\y...,,


In [4]:
print("Visual groups folder:", DETECTION_OUT)
print("Expected: about 24 PNG files, if every landmark has at least 2 query images.")
print("Manual review file:", review_csv)

Visual groups folder: d:\CODE\PY\BJTU_Vision\outputs\detection_results\visual_groups
Expected: about 24 PNG files, if every landmark has at least 2 query images.
Manual review file: d:\CODE\PY\BJTU_Vision\outputs\detection_results\manual_review.csv
